### Verification for entry and exit times

In [3]:
import geopandas as gpd
from shapely.geometry import LineString, Point
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
from geopy.distance import geodesic

# Set the current working directory
import os
os.chdir("/home/jovyan/work/Typhoon_IBF_Rice_Damage_Model/")
cdir = os.getcwd()

# Load the typhoon tracks
tracks_path = 'IBF_typhoon_model/data/gis_data/typhoon_tracks/tracks_filtered.shx'
typhoon_tracks = gpd.read_file(tracks_path)

# Load the municipal borders
municipalities_path = 'IBF_typhoon_model/data/phl_administrative_boundaries/phl_admbnda_adm3.shx'
municipalities = gpd.read_file(municipalities_path)

# Convert datetime columns to actual datetime objects if not already done
typhoon_tracks['datetime'] = pd.to_datetime(typhoon_tracks[['year', 'month', 'day', 'hour']].astype(str).agg('-'.join, axis=1), format='%Y-%m-%d-%H')

# Function to fill NA values with the last available value for USA_ROCI
def fill_roci(track):
    track['USA_ROCI'] = track['USA_ROCI'].fillna(method='ffill').fillna(0)
    return track

# Function to create a circle in kilometers around a point
def create_circle_km(ax, center_point, radius_km, **kwargs):
    lat_radius = radius_km / 111  # Convert km to degrees of latitude
    circle = plt.Circle((center_point.x, center_point.y), lat_radius, **kwargs)
    ax.add_patch(circle)

# Function to visualize ROCI for the entire typhoon duration across the whole Philippines
def visualize_all_roci(typhoon_id, target_municipality_pcode):
    track = typhoon_tracks[typhoon_tracks['SID'] == typhoon_id]
    track = fill_roci(track)
    target_municipality = municipalities[municipalities["ADM3_PCODE"] == target_municipality_pcode]

    fig, ax = plt.subplots(figsize=(10, 10))
    municipalities.boundary.plot(ax=ax, color='black', linewidth=0.5)
    target_municipality.boundary.plot(ax=ax, color='green', linewidth=2)
    track.plot(ax=ax, color='red', linewidth=1)
    
    for i in range(len(track)):
        segment = track.iloc[i].geometry
        if isinstance(segment, LineString):
            centroid = segment.centroid
            roci_nm = track.iloc[i]['USA_ROCI']
            if roci_nm > 0:
                # Convert nautical miles to kilometers (1 NM = 1.852 km)
                roci_km = roci_nm * 1.852
                create_circle_km(ax, centroid, roci_km, color='blue', alpha=0.3)
    
    plt.title(f'Typhoon ROCI Visualization for Typhoon ID: {typhoon_id}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

# Function to visualize ROCI at a specific datetime across the whole Philippines
def visualize_roci_time_step(typhoon_id, datetime, target_municipality_pcode):
    track = typhoon_tracks[typhoon_tracks['SID'] == typhoon_id]
    track = fill_roci(track)
    target_municipality = municipalities[municipalities["ADM3_PCODE"] == target_municipality_pcode]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    municipalities.boundary.plot(ax=ax, color='black', linewidth=0.5)
    target_municipality.boundary.plot(ax=ax, color='green', linewidth=2)
    track.plot(ax=ax, color='red', linewidth=1)
    
    row = track[track['datetime'] == datetime]
    if not row.empty:
        segment = row.iloc[0].geometry
        if isinstance(segment, LineString):
            centroid = segment.centroid
            roci_nm = row.iloc[0]['USA_ROCI']
            if roci_nm > 0:
                # Convert nautical miles to kilometers (1 NM = 1.852 km)
                roci_km = roci_nm * 1.852
                create_circle_km(ax, centroid, roci_km, color='blue', alpha=0.3)
    
    plt.title(f'Typhoon ROCI at {datetime} for Typhoon ID: {typhoon_id}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

# Function to visualize ROCI at a specific datetime zoomed into the target municipality
def visualize_roci_time_step_zoomed(typhoon_id, datetime, target_municipality_pcode):
    track = typhoon_tracks[typhoon_tracks['SID'] == typhoon_id]
    track = fill_roci(track)
    target_municipality = municipalities[municipalities["ADM3_PCODE"] == target_municipality_pcode]
    
    fig, ax = plt.subplots(figsize=(10, 10))
    target_municipality.boundary.plot(ax=ax, color='black', linewidth=1)
    track.plot(ax=ax, color='red', linewidth=1)
    
    row = track[track['datetime'] == datetime]
    if not row.empty:
        segment = row.iloc[0].geometry
        if isinstance(segment, LineString):
            centroid = segment.centroid
            roci_nm = row.iloc[0]['USA_ROCI']
            if roci_nm > 0:
                # Convert nautical miles to kilometers (1 NM = 1.852 km)
                roci_km = roci_nm * 1.852
                create_circle_km(ax, centroid, roci_km, color='blue', alpha=0.3)
    
    plt.title(f'Typhoon ROCI at {datetime} for Typhoon ID: {typhoon_id}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    
    # Zoom in to the target municipality
    bounds = target_municipality.geometry.total_bounds
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    
    plt.show()

# Create an interactive slider for datetime values
def update_datetime_widget(typhoon_id):
    track = typhoon_tracks[typhoon_tracks['SID'] == typhoon_id]
    datetime_options = track['datetime'].unique()
    return widgets.SelectionSlider(
        options=datetime_options,
        description='Datetime:',
        continuous_update=False,
        orientation='horizontal',
        layout={'width': '800px'}
    )

# Function to display all visualizations
def display_all_visualizations(typhoon_id, target_municipality_pcode):
    def update(datetime):
        visualize_all_roci(typhoon_id, target_municipality_pcode)
        visualize_roci_time_step(typhoon_id, datetime, target_municipality_pcode)
        visualize_roci_time_step_zoomed(typhoon_id, datetime, target_municipality_pcode)
    
    datetime_slider = update_datetime_widget(typhoon_id)
    interact(update, datetime=datetime_slider)

# Create a dropdown for typhoon IDs and a text box for the municipality code
typhoon_ids = typhoon_tracks['SID'].unique()

typhoon_id_dropdown = widgets.Dropdown(options=typhoon_ids, description='Typhoon ID:')
municipality_code_text = widgets.Text(value='PH037706000', description='ADM3 Code:')

# Function to combine typhoon ID and municipality code inputs
def combined_inputs(typhoon_id, target_municipality_pcode):
    display_all_visualizations(typhoon_id, target_municipality_pcode)

interact(combined_inputs, typhoon_id=typhoon_id_dropdown, target_municipality_pcode=municipality_code_text)


interactive(children=(Dropdown(description='Typhoon ID:', options=('2006329N06150', '2009268N14128', '2009270N…

<function __main__.combined_inputs(typhoon_id, target_municipality_pcode)>

### Visualize Tracks with extents and damages
Noteable finding is that Typhoon Melor2015 (2015344N07145),is the only one for which the extents dont overlap with all the damages.


In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.colors as mcolors

# Set the current working directory
import os
os.chdir("/home/jovyan/work/Typhoon_IBF_Rice_Damage_Model/")
cdir = os.getcwd()

# Load the typhoon tracks
tracks_path = 'IBF_typhoon_model/data/gis_data/typhoon_tracks/tracks_filtered.shx'
typhoon_tracks = gpd.read_file(tracks_path)

# Load the municipal borders
municipalities_path = 'IBF_typhoon_model/data/phl_administrative_boundaries/phl_admbnda_adm3.shx'
municipalities = gpd.read_file(municipalities_path)

# Load the loss data
loss_data_path = r'IBF_typhoon_model/data/restricted_data/combined_input_data/input_data_05.xlsx'
loss_data = pd.read_excel(loss_data_path)

# Convert datetime columns to actual datetime objects if not already done
typhoon_tracks['datetime'] = pd.to_datetime(typhoon_tracks[['year', 'month', 'day', 'hour']].astype(str).agg('-'.join, axis=1), format='%Y-%m-%d-%H')

# Function to fill NA values with the last available value for USA_ROCI
def fill_roci(track):
    track['USA_ROCI'] = track['USA_ROCI'].fillna(method='ffill').fillna(0)
    return track

# Function to merge loss data with municipalities based on the selected SID
def merge_loss_data(municipalities, loss_data, storm_id):
    loss_data_filtered = loss_data[loss_data['storm_id'] == storm_id]
    municipalities_with_loss = municipalities.merge(loss_data_filtered, left_on='ADM3_PCODE', right_on='mun_code', how='left')
    return municipalities_with_loss

# Function to create a circle in kilometers around a point
def create_circle_km(ax, center_point, radius_km, **kwargs):
    lat_radius = radius_km / 111  # Convert km to degrees of latitude
    circle = plt.Circle((center_point.x, center_point.y), lat_radius, **kwargs)
    ax.add_patch(circle)

# Function to visualize typhoon tracks, extents, and damages
def visualize_typhoon_with_damage(typhoon_id):
    track = typhoon_tracks[typhoon_tracks['SID'] == typhoon_id]
    track = fill_roci(track)
    municipalities_with_loss = merge_loss_data(municipalities, loss_data, typhoon_id)

    fig, axs = plt.subplots(1, 2, figsize=(20, 10))

    # Plot 1: Municipalities with extents and tracks
    municipalities_with_loss.boundary.plot(ax=axs[0], color='black', linewidth=0.5)
    municipalities_with_loss.plot(ax=axs[0], column='perc_loss', cmap='viridis', legend=True, alpha=0.5, 
                                  missing_kwds={'color': 'lightgrey', "label": "No data"})
    track.plot(ax=axs[0], color='red', linewidth=1)
    
    for i in range(len(track)):
        segment = track.iloc[i].geometry
        if isinstance(segment, LineString):
            centroid = segment.centroid
            roci_nm = track.iloc[i]['USA_ROCI']
            if roci_nm > 0:
                # Convert nautical miles to kilometers (1 NM = 1.852 km)
                roci_km = roci_nm * 1.852
                create_circle_km(axs[0], centroid, roci_km, color='blue', alpha=0.3)

    # Set plot limits to the extent of the Philippines
    ph_bounds = municipalities.total_bounds
    axs[0].set_xlim(ph_bounds[0], ph_bounds[2])
    axs[0].set_ylim(ph_bounds[1], ph_bounds[3])

    axs[0].set_title(f'Typhoon Tracks and Extents for Typhoon ID: {typhoon_id}')
    axs[0].set_xlabel('Longitude')
    axs[0].set_ylabel('Latitude')

    # Plot 2: Municipalities with damage data only
    municipalities_with_loss.boundary.plot(ax=axs[1], color='black', linewidth=0.5)
    municipalities_with_loss.plot(ax=axs[1], column='perc_loss', cmap='viridis', legend=True, alpha=0.5, 
                                  missing_kwds={'color': 'lightgrey', "label": "No data"})
    
    # Set plot limits to the extent of the Philippines
    axs[1].set_xlim(ph_bounds[0], ph_bounds[2])
    axs[1].set_ylim(ph_bounds[1], ph_bounds[3])

    axs[1].set_title('Municipalities with Damage Data (Perc Loss)')
    axs[1].set_xlabel('Longitude')
    axs[1].set_ylabel('Latitude')

    plt.tight_layout()
    plt.show()

# Create a dropdown for typhoon IDs
typhoon_ids = typhoon_tracks['SID'].unique()

typhoon_id_dropdown = widgets.Dropdown(options=typhoon_ids, description='Typhoon ID:')

# Function to update the visualization based on the selected typhoon ID
def update_visualization(typhoon_id):
    visualize_typhoon_with_damage(typhoon_id)

# Create the interactive widget
interact(update_visualization, typhoon_id=typhoon_id_dropdown)


ModuleNotFoundError: No module named 'geopandas'

### Statistical analysis into time difference between entry and exit datetime

In [10]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact

# Change to the specified directory
os.chdir("/home/jovyan/work/Typhoon_IBF_Rice_Damage_Model/")
cdir = os.getcwd()

# Load the original CSV file
file_path = os.path.join(cdir, 'IBF_typhoon_model/data/gis_data/typhoon_enter_exit_per_municipality.csv')
df = pd.read_csv(file_path)

# Ensure the datetime columns are in datetime format
df['entry_date'] = pd.to_datetime(df['entry_date'])
df['exit_date'] = pd.to_datetime(df['exit_date'])

# Load the data overview file
overview_path = os.path.join(cdir, 'IBF_typhoon_model/data/data_overview.xlsx')
overview_df = pd.read_excel(overview_path, sheet_name='typhoon_overview')

# Extract the relevant year information
overview_df['year'] = overview_df['name_year'].str.extract(r'(\d{4})').astype(int)

# Merge the original data with the overview data to include the year
df = df.merge(overview_df[['name_year', 'year']], left_on='name_year', right_on='name_year', how='left')

# Function to filter data based on year and calculate statistics
def process_data(pre_2016=True, save_csv=False):
    # Filter based on year
    if pre_2016:
        filtered_df = df[df['year'] < 2016]
    else:
        filtered_df = df

    # Calculate the difference in hours between exit and entry
    filtered_df['hours_difference'] = (filtered_df['exit_date'] - filtered_df['entry_date']).dt.total_seconds() / 3600

    # Save the modified dataframe to a new CSV file if requested
    if save_csv:
        output_file_path = os.path.join(cdir, 'IBF_typhoon_model/data/gis_data/typhoon_enter_exit_per_municipality_with_hours_difference_filtered.csv')
        filtered_df.to_csv(output_file_path, index=False)
        print(f"CSV file with hours difference saved to {output_file_path}")

    # Plot a histogram of the hours difference
    plt.figure(figsize=(10, 6))
    plt.hist(filtered_df['hours_difference'], bins=30, edgecolor='black')
    plt.title('Distribution of Hours Difference Between Entry and Exit')
    plt.xlabel('Hours Difference')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

    # Calculate and print statistics of the hours difference
    mean_hours = filtered_df['hours_difference'].mean()
    median_hours = filtered_df['hours_difference'].median()
    std_hours = filtered_df['hours_difference'].std()
    min_hours = filtered_df['hours_difference'].min()
    max_hours = filtered_df['hours_difference'].max()

    print(f"Mean Hours Difference: {mean_hours}")
    print(f"Median Hours Difference: {median_hours}")
    print(f"Standard Deviation: {std_hours}")
    print(f"Minimum Hours Difference: {min_hours}")
    print(f"Maximum Hours Difference: {max_hours}")

    # Plot a histogram of the hours difference between 0 and 50 hours
    plt.figure(figsize=(10, 6))

    # Define the bins to be multiples of 3 between 0 and 50
    bins = np.arange(0, 51, 3)

    # Filtered DataFrame for hours difference between 0 and 50
    filtered_0_50_df = filtered_df[(filtered_df['hours_difference'] >= 0) & (filtered_df['hours_difference'] <= 50)]

    plt.hist(filtered_0_50_df['hours_difference'], bins=bins, edgecolor='black')
    plt.title('Distribution of Hours Difference Between Entry and Exit (0 to 50 hours)')
    plt.xlabel('Hours Difference (0 to 50, in multiples of 3)')
    plt.ylabel('Frequency')
    plt.xticks(bins)
    plt.grid(True)
    plt.show()

    # Calculate and print statistics of the hours difference between 0 and 50 hours
    mean_hours_filtered = filtered_0_50_df['hours_difference'].mean()
    median_hours_filtered = filtered_0_50_df['hours_difference'].median()
    std_hours_filtered = filtered_0_50_df['hours_difference'].std()
    min_hours_filtered = filtered_0_50_df['hours_difference'].min()
    max_hours_filtered = filtered_0_50_df['hours_difference'].max()

    print(f"Mean Hours Difference (0 to 50 hours): {mean_hours_filtered}")
    print(f"Median Hours Difference (0 to 50 hours): {median_hours_filtered}")
    print(f"Standard Deviation (0 to 50 hours): {std_hours_filtered}")
    print(f"Minimum Hours Difference (0 to 50 hours): {min_hours_filtered}")
    print(f"Maximum Hours Difference (0 to 50 hours): {max_hours_filtered}")

# Create interactive widgets for filtering and saving options
pre_2016_toggle = widgets.Checkbox(value=True, description='Filter Pre-2016 Data')
save_csv_toggle = widgets.Checkbox(value=False, description='Save CSV')

# Create the interactive widget for the processing function
interact(process_data, pre_2016=pre_2016_toggle, save_csv=save_csv_toggle)


interactive(children=(Checkbox(value=True, description='Filter Pre-2016 Data'), Checkbox(value=False, descript…

<function __main__.process_data(pre_2016=True, save_csv=False)>